# DeepBait Model Demo

This notebook shows headline generation with one representative checkpoint for each model family in the project:
- `checkpoints/exp1_direct/best_model.pt`
- `checkpoints/exp2_finetune/best_model.pt`
- `checkpoints/exp3_bart/best_model`

The paths are hardcoded on purpose for demo use.


In [1]:
from pathlib import Path
import sys

import pandas as pd
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

ROOT = Path(r"D:/Documents/repos/dpbt-2")
SRC_DIR = ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from generate import generate_batch, load_model

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DIRECT_LSTM_PATH = ROOT / "checkpoints" / "exp1_direct" / "best_model.pt"
FINETUNE_LSTM_PATH = ROOT / "checkpoints" / "exp2_finetune" / "best_model.pt"
BART_MODEL_DIR = ROOT / "checkpoints" / "exp3_bart" / "best_model"

sample_article = """
Scientists at MIT have developed a new artificial intelligence system that can diagnose certain types of cancer from blood samples with accuracy comparable to experienced physicians. The research, published in Nature Medicine, used deep learning to analyze protein biomarkers in more than 10,000 patient samples. The team says the method could reduce the time needed for early screening and help hospitals offer lower-cost testing in routine clinical settings.
""".strip()

print(f"Device: {DEVICE}")
print(f"Direct LSTM checkpoint      : {DIRECT_LSTM_PATH}")
print(f"Pretrain+finetune checkpoint: {FINETUNE_LSTM_PATH}")
print(f"BART model dir              : {BART_MODEL_DIR}")


Device: cuda
Direct LSTM checkpoint      : D:\Documents\repos\dpbt-2\checkpoints\exp1_direct\best_model.pt
Pretrain+finetune checkpoint: D:\Documents\repos\dpbt-2\checkpoints\exp2_finetune\best_model.pt
BART model dir              : D:\Documents\repos\dpbt-2\checkpoints\exp3_bart\best_model


In [2]:
print("Sample article:")
print(sample_article)


Sample article:
Scientists at MIT have developed a new artificial intelligence system that can diagnose certain types of cancer from blood samples with accuracy comparable to experienced physicians. The research, published in Nature Medicine, used deep learning to analyze protein biomarkers in more than 10,000 patient samples. The team says the method could reduce the time needed for early screening and help hospitals offer lower-cost testing in routine clinical settings.


In [3]:
def run_lstm_demo(model_name: str, checkpoint_path: Path, temperature: float = 0.8, num_headlines: int = 3) -> list[str]:
    model, word2idx, idx2word, max_article_len = load_model(str(checkpoint_path), DEVICE)
    torch.manual_seed(42)
    headlines = generate_batch(
        model=model,
        word2idx=word2idx,
        idx2word=idx2word,
        article_text=sample_article,
        num_headlines=num_headlines,
        max_article_len=max_article_len,
        max_len=20,
        temperature=temperature,
        device=DEVICE,
    )
    print(model_name)
    for idx, headline in enumerate(headlines, start=1):
        print(f"  {idx}. {headline}")
    print()
    return headlines


In [4]:
direct_headlines = run_lstm_demo(
    model_name="Direct LSTM",
    checkpoint_path=DIRECT_LSTM_PATH,
)


Direct LSTM
  1. how many in 2016 for fun
  2. i am my portraits to cook but to always the experts
  3. 7 stunning sheets to tesla turned in a year



In [5]:
finetune_headlines = run_lstm_demo(
    model_name="Pretrain + Fine-tune LSTM",
    checkpoint_path=FINETUNE_LSTM_PATH,
)


Pretrain + Fine-tune LSTM
  1. how many americans may catch for life in the us
  2. how to cook little spiders
  3. the new disease and brain drug is known as disease



In [6]:
bart_tokenizer = AutoTokenizer.from_pretrained(BART_MODEL_DIR)
bart_model = AutoModelForSeq2SeqLM.from_pretrained(BART_MODEL_DIR).to(DEVICE)
bart_model.generation_config.max_length = None
bart_model.generation_config.min_length = 1
bart_model.eval()

def generate_bart_headline(article: str, num_beams: int = 4, max_input_len: int = 512, max_output_len: int = 32) -> str:
    inputs = bart_tokenizer(
        article,
        return_tensors="pt",
        max_length=max_input_len,
        truncation=True,
        return_token_type_ids=False,
    ).to(DEVICE)

    noise_strings = ["#", "@", " #", " @"]
    bad_words_ids = [bart_tokenizer.encode(text, add_special_tokens=False) for text in noise_strings]
    bad_words_ids = [ids for ids in bad_words_ids if ids]

    with torch.no_grad():
        output_ids = bart_model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            num_beams=num_beams,
            min_length=1,
            max_new_tokens=max_output_len,
            no_repeat_ngram_size=3,
            bad_words_ids=bad_words_ids,
            early_stopping=True,
        )

    headline = bart_tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return headline.split("\n")[0].strip()

bart_headline = generate_bart_headline(sample_article)
print("BART")
print(f"  1. {bart_headline}")


Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

BART
  1. MIT develops AI system that can diagnose certain types of cancer from blood samples


In [7]:
comparison = pd.DataFrame(
    [
        {"model": "Direct LSTM", "headline": direct_headlines[0]},
        {"model": "Pretrain + Fine-tune LSTM", "headline": finetune_headlines[0]},
        {"model": "BART", "headline": bart_headline},
    ]
)

comparison


,model,headline
0,Direct LSTM,how many in 2016 for fun
1,Pretrain + Fine-tune LSTM,how many americans may catch for life in the us
2,BART,MIT develops AI system that can diagnose certa...
